In [ ]:
JSON = flexible, nested, unpredictable
{
  "emp_id": 1,
  "name": "Alice",
  "address": {              ← nested object!
    "city": "NYC",
    "zip": {                ← nested inside nested!
      "code": "10001",
      "area": "Manhattan"
    }
  },
  "skills": ["Python","SQL"] ← array!
  "projects": [             ← array of objects!
    {"id":1, "name":"ETL"},
    {"id":2, "name":"ML"}
  ]
}
Different records can have DIFFERENT fields!
Spark has to figure out structure! 😵

🎯 Types of JSON — Each Needs Different Approach!
Type 1 — JSON Lines (one record per line)
{"emp_id":1,"name":"Alice"}
{"emp_id":2,"name":"Bob"}
↑ Most common, easiest for Spark!

Type 2 — Pretty printed / Multiline
{
  "emp_id": 1,
  "name": "Alice"
}
↑ Needs special option!

Type 3 — JSON Array
[
  {"emp_id":1,"name":"Alice"},
  {"emp_id":2,"name":"Bob"}
]
↑ Spark can't read directly! Needs handling!

Type 4 — Nested JSON
{"emp_id":1,"address":{"city":"NYC"}}
↑ Needs explode/flattening!

Type 5 — Mixed/Inconsistent fields
{"emp_id":1,"name":"Alice","salary":80000}
{"emp_id":2,"name":"Bob"}              ← salary missing!
{"emp_id":3,"age":25}                  ← totally different field!
↑ Schema inference nightmare!

1️⃣ JSON Lines — Simplest!
python# data/emp.json
# {"emp_id":1,"name":"Alice","salary":80000}
# {"emp_id":2,"name":"Bob","salary":90000}

df = spark.read \
    .format("json") \
    .load("data/emp.json")

df.show()
# +------+-----+------+
# |emp_id| name|salary|
# +------+-----+------+
# |     1|Alice| 80000|
# |     2|  Bob| 90000|

df.printSchema()
# root
#  |-- emp_id: long
#  |-- name: string
#  |-- salary: long

2️⃣ Multiline JSON — Pretty Printed!
python# data/emp.json
# {
#   "emp_id": 1,
#   "name": "Alice"
# }

# ❌ Without multiLine — reads garbage!
df = spark.read.json("data/emp.json")
# each LINE read as separate record
# broken JSON on each line → nulls everywhere!

# ✅ With multiLine — reads correctly!
df = spark.read \
    .format("json") \
    .option("multiLine", "true") \   # ← KEY option!
    .load("data/emp.json")

df.show()
# +------+-----+
# |emp_id| name|
# +------+-----+
# |     1|Alice|

3️⃣ Nested JSON — The Real Challenge!
python# data/nested.json
# {"emp_id":1,"name":"Alice",
#  "address":{"city":"NYC","zip":"10001"},
#  "skills":["Python","SQL","Spark"]}

df = spark.read.json("data/nested.json")

df.printSchema()
# root
#  |-- emp_id: long
#  |-- name: string
#  |-- address: struct          ← nested struct!
#  |    |-- city: string
#  |    |-- zip: string
#  |-- skills: array            ← array!
#  |    |-- element: string

df.show(truncate=False)
# +------+-----+---------------+------------------+
# |emp_id|name |address        |skills            |
# +------+-----+---------------+------------------+
# |1     |Alice|{NYC, 10001}   |[Python, SQL, Spark]

# Access nested fields
df.select(
    "emp_id",
    "name",
    "address.city",        # dot notation for struct!
    "address.zip"
).show()
# +------+-----+----+-----+
# |emp_id| name|city|  zip|
# +------+-----+----+-----+
# |     1|Alice| NYC|10001|

4️⃣ Flatten Nested JSON — Production Pattern!
pythonfrom pyspark.sql.functions import col, explode

df = spark.read.json("data/nested.json")

# Flatten struct
df_flat = df.select(
    col("emp_id"),
    col("name"),
    col("address.city").alias("city"),    # flatten struct!
    col("address.zip").alias("zip")
)

# Explode array → one row per skill!
df_skills = df.select(
    col("emp_id"),
    col("name"),
    explode(col("skills")).alias("skill")  # array → rows!
)

df_skills.show()
# +------+-----+------+
# |emp_id| name| skill|
# +------+-----+------+
# |     1|Alice|Python|   ← one row per skill!
# |     1|Alice|   SQL|
# |     1|Alice| Spark|

5️⃣ Array of Objects — Complex Nested!
python# {"emp_id":1,
#  "projects":[
#    {"proj_id":1,"proj_name":"ETL"},
#    {"proj_id":2,"proj_name":"ML"}
#  ]}

df = spark.read.json("data/projects.json")

df.printSchema()
# |-- projects: array
# |    |-- element: struct
# |    |    |-- proj_id: long
# |    |    |-- proj_name: string

# Explode array of structs!
from pyspark.sql.functions import explode, col

df_exploded = df.select(
    col("emp_id"),
    explode(col("projects")).alias("project")  # array → rows
)

# Now access struct fields
df_final = df_exploded.select(
    col("emp_id"),
    col("project.proj_id"),
    col("project.proj_name")
)

df_final.show()
# +------+-------+---------+
# |emp_id|proj_id|proj_name|
# +------+-------+---------+
# |     1|      1|      ETL|  ← one row per project!
# |     1|      2|       ML|

6️⃣ Missing Fields — Handle Inconsistency!
python# {"emp_id":1,"name":"Alice","salary":80000}
# {"emp_id":2,"name":"Bob"}              ← no salary!
# {"emp_id":3,"salary":70000}            ← no name!

df = spark.read.json("data/inconsistent.json")

df.show()
# +------+-----+------+
# |emp_id| name|salary|
# +------+-----+------+
# |     1|Alice| 80000|
# |     2|  Bob|  null|  ← salary = null ✅
# |     3| null| 70000|  ← name = null ✅

# Spark fills missing fields with null automatically!

🎯 All Key JSON Options!
pythondf = spark.read \
    .format("json") \
    .option("multiLine",             "true") \  # pretty JSON
    .option("allowComments",         "true") \  # allow // comments
    .option("allowUnquotedFieldNames","true") \  # {name: "Alice"}
    .option("allowSingleQuotes",     "true") \  # {'name':'Alice'}
    .option("allowNumericLeadingZeros","true") \ # 007
    .option("mode", "PERMISSIVE") \             # nulls for bad rows
    .option("mode", "DROPMALFORMED") \          # skip bad rows
    .option("mode", "FAILFAST") \               # crash on bad row
    .option("columnNameOfCorruptRecord","_corrupt") \ # catch bad!
    .load("data/emp.json")

format("json") → reads JSON FILE
                 entire file is JSON
                 Spark parses whole file!

from_json()    → reads JSON STRING
                 inside a DataFrame COLUMN
                 one cell contains JSON text!
Real world scenarios:

1. Kafka Streaming → messages come as JSON strings
2. API responses  → stored as raw JSON in DB column
3. Log files      → each log line has JSON payload
4. format("text") → read JSON file as raw strings

DataFrame looks like:
+------+------------------------------------------+
|msg_id|                                   payload|
+------+------------------------------------------+
|     1|{"emp_id":1,"name":"Alice","salary":80000}|  ← JSON string!
|     2|{"emp_id":2,"name":"Bob","salary":90000}  |  ← J

data = [
    (1, '{"emp_id":1,"name":"Alice","salary":80000}'),
    (2, '{"emp_id":2,"name":"Bob","salary":90000}'),
    (3, '{"emp_id":3,"name":"Carol","salary":70000}')
]

df = spark.createDataFrame(data, ["msg_id", "payload"])

df.show(truncate=False)
# +------+------------------------------------------+
# |msg_id|payload                                   |
# +------+------------------------------------------+
# |1     |{"emp_id":1,"name":"Alice","salary":80000}|
# |2     |{"emp_id":2,"name":"Bob","salary":90000}  |
# |3     |{"emp_id":3,"name":"Carol","salary":70000}|

# Step 1 — Define schema of JSON inside column
schema = StructType([
    StructField("emp_id", IntegerType(), True),
    StructField("name",   StringType(),  True),
    StructField("salary", IntegerType(), True)
])

# Step 2 — Parse JSON string column!
df_parsed = df.withColumn(
    "parsed",
    from_json(col("payload"), schema)   # ← parse string → struct!
)

df_parsed.printSchema()
# root
#  |-- msg_id: long
#  |-- payload: string
#  |-- parsed: struct          ← struct created!
#  |    |-- emp_id: integer
#  |    |-- name: string
#  |    |-- salary: integer

format("json")              from_json()
──────────────              ───────────
File level                  Column level
Reads JSON file             Parses JSON string in column
Entire file is JSON         One column contains JSON
Used at spark.read          Used in withColumn/select
No schema needed*           Schema REQUIRED!
Output = DataFrame          Output = Struct column

spark.read                  df.withColumn("parsed",
  .format("json")             from_json(col("payload"),
  .load("file.json")                    schema))

df_complex_13=df_complex.select("name",to_json(col("payroll")).alias("payroll_json"))----to json

Schema type     What you see         How to work with it
───────────     ────────────         ───────────────────
struct          {60000, 15000}       col("payroll.basic")
array           [Python, SQL]        explode(col("skills"))
map             {key → value}        col("map.key")
string(json)    {"basic":60000}      from_json(col, schema)
string(normal)  "Alice"              col("name")
string(date)    "2020-01-15"         to_date(col, format)